# Code 8.5.3: Generating from the mini GPT

Loads the mini GPT saved by Code 8.4.2 and generates from the prompt "ROMEO:" with greedy decoding, top-p sampling, and temperature 1.5. The loop has no cache and feeds at most the last $n_{\max}$ tokens.

This listing loads `mini_gpt.pt`, which Code 8.4.2 writes after pretraining the mini GPT.


In [ ]:
from pathlib import Path

# Code 8.2.1, saved as gpt.py so `from gpt import ...` works in this notebook.
_GPT_PY = 'import math\nfrom dataclasses import dataclass\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n@dataclass\nclass GPTConfig:\n    V: int = 50257          # vocabulary size\n    n_max: int = 1024       # maximum context length\n    N: int = 12             # number of blocks\n    d: int = 768            # model width\n    h: int = 12             # heads\n    dropout: float = 0.0\n\nclass CausalSelfAttention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.qkv = nn.Linear(cfg.d, 3 * cfg.d)          # W_Q, W_K, W_V in one matrix\n        self.proj = nn.Linear(cfg.d, cfg.d)             # W_O\n        self.h, self.dropout = cfg.h, cfg.dropout\n    def forward(self, x):\n        B, n, d = x.shape\n        q, k, v = self.qkv(x).split(d, dim=-1)\n        q, k, v = (t.view(B, n, self.h, d // self.h).transpose(1, 2) for t in (q, k, v))\n        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,   # the causal mask of Section 6.4\n                                           dropout_p=self.dropout if self.training else 0.0)\n        return self.proj(y.transpose(1, 2).reshape(B, n, d))\n\nclass MLP(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.fc = nn.Linear(cfg.d, 4 * cfg.d)\n        self.proj = nn.Linear(4 * cfg.d, cfg.d)\n    def forward(self, x):\n        return self.proj(F.gelu(self.fc(x), approximate="tanh"))   # GPT-2 uses the tanh form of GELU\n\nclass Block(nn.Module):                                 # pre-norm decoder block, no cross-attention\n    def __init__(self, cfg):\n        super().__init__()\n        self.ln1, self.attn = nn.LayerNorm(cfg.d), CausalSelfAttention(cfg)\n        self.ln2, self.mlp = nn.LayerNorm(cfg.d), MLP(cfg)\n        self.drop = nn.Dropout(cfg.dropout)\n    def forward(self, x):\n        x = x + self.drop(self.attn(self.ln1(x)))\n        return x + self.drop(self.mlp(self.ln2(x)))\n\nclass GPT(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.tok = nn.Embedding(cfg.V, cfg.d)\n        self.pos = nn.Embedding(cfg.n_max, cfg.d)       # learned absolute positions\n        self.drop = nn.Dropout(cfg.dropout)\n        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.N))\n        self.ln_f = nn.LayerNorm(cfg.d)                 # final LayerNorm after the last block\n        self.apply(self._init)\n        for name, p in self.named_parameters():         # residual branches: std / sqrt(2N)\n            if name.endswith("proj.weight"):\n                nn.init.normal_(p, std=0.02 / math.sqrt(2 * cfg.N))\n    def _init(self, m):\n        if isinstance(m, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(m.weight, std=0.02)\n        if isinstance(m, nn.Linear) and m.bias is not None:\n            nn.init.zeros_(m.bias)\n    def forward(self, idx, targets=None):\n        n = idx.size(1)\n        assert n <= self.cfg.n_max\n        x = self.drop(self.tok(idx) + self.pos(torch.arange(n, device=idx.device)))\n        for block in self.blocks:\n            x = block(x)\n        logits = self.ln_f(x) @ self.tok.weight.T       # output projection tied to the embedding\n        if targets is None:\n            return logits\n        return logits, F.cross_entropy(logits.flatten(0, 1), targets.flatten())\n'
Path('gpt.py').write_text(_GPT_PY)


In [ ]:
import tiktoken
import torch
from gpt import GPT                                      # Code 8.2.1
from sampling import sample_next                        # Code 8.5.1

ckpt = torch.load("mini_gpt.pt", weights_only=False)   # saved by Code 8.4.2
mini = GPT(ckpt["cfg"]).eval()
mini.load_state_dict(ckpt["model"])
vocab, enc = ckpt["vocab"], tiktoken.get_encoding("gpt2")

@torch.no_grad()
def generate_mini(prompt, max_new=40, seed=0, **kw):
    g = torch.Generator().manual_seed(seed)
    x = torch.searchsorted(vocab, torch.tensor(enc.encode(prompt)))[None]   # GPT-2 IDs -> model IDs
    for _ in range(max_new):
        logits = mini(x[:, -mini.cfg.n_max:])[0, -1]   # no cache: rerun the last n_max tokens
        x = torch.cat([x, torch.tensor([[sample_next(logits, generator=g, **kw)]])], dim=1)
    return enc.decode(vocab[x[0]].tolist())

for name, kw in [("greedy", dict(temperature=0)), ("top-p 0.9", dict(top_p=0.9)),
                 ("temperature 1.5", dict(temperature=1.5))]:
    print(f"--- {name}")
    print(generate_mini("ROMEO:\n", **kw))
# --- greedy
# ROMEO:
# I'll be a man,
# And I have not,
# And I have not the king,
# And I have not the king,
# And I have not the king,
# And I have
# --- top-p 0.9
# ROMEO:
# KING HENRY VI:
# I so Clifford! and be what I say he's the prince.
# Let me too is sir; but pity?
# How, is ourost that speak's
# --- temperature 1.5
# ROMEO:
# KINGSound trumpet for the raised together from soguby and be ancestors laceBR treasury dram nothing? yet would scared poison
# But sir on him pity thy wordsVill sh guess figure of that speak's
